In [1]:
from sympy import Array, Symbol, tensorcontraction, tensorproduct
import black


def bending_function_sympy(Q, Bp, Bpp):
    QM = Q.reshape(4, 3).transpose()

    # Projections (3-vector each)

    yip = tensorcontraction(tensorproduct(QM, Bp), (1, 2))
    yipp = tensorcontraction(tensorproduct(QM, Bpp), (1, 2))

    def cross(a: Array, b: Array):
        """
        3d cross product
        https://reference.wolfram.com/language/ref/Cross.html
        """
        return Array([
            a[1] * b[2] - a[2] * b[1],
            a[2] * b[0] - a[0] * b[2],
            a[0] * b[1] - a[1] * b[0],
        ])

    def norm3(arr: Array):
        x, y, z = arr[0], arr[1], arr[2]
        return (x**2 + y**2 + z**2) ** (0.5)

    return (norm3(cross(yip, yipp)) / (norm3(yip) ** 3)) ** 2


sympy_res = bending_function_sympy(
    Array([Symbol(f"q{i}") for i in range(1, 13)]),
    Array([Symbol(f"bp{i}") for i in range(1, 5)]),
    Array([Symbol(f"bpp{i}") for i in range(1, 5)]),
)
print(black.format_str(repr(sympy_res), mode=black.Mode(line_length=88)).strip())

(
    (
        (bp1 * q1 + bp2 * q4 + bp3 * q7 + bp4 * q10)
        * (bpp1 * q2 + bpp2 * q5 + bpp3 * q8 + bpp4 * q11)
        - (bp1 * q2 + bp2 * q5 + bp3 * q8 + bp4 * q11)
        * (bpp1 * q1 + bpp2 * q4 + bpp3 * q7 + bpp4 * q10)
    )
    ** 2
    + (
        -(bp1 * q1 + bp2 * q4 + bp3 * q7 + bp4 * q10)
        * (bpp1 * q3 + bpp2 * q6 + bpp3 * q9 + bpp4 * q12)
        + (bp1 * q3 + bp2 * q6 + bp3 * q9 + bp4 * q12)
        * (bpp1 * q1 + bpp2 * q4 + bpp3 * q7 + bpp4 * q10)
    )
    ** 2
    + (
        (bp1 * q2 + bp2 * q5 + bp3 * q8 + bp4 * q11)
        * (bpp1 * q3 + bpp2 * q6 + bpp3 * q9 + bpp4 * q12)
        - (bp1 * q3 + bp2 * q6 + bp3 * q9 + bp4 * q12)
        * (bpp1 * q2 + bpp2 * q5 + bpp3 * q8 + bpp4 * q11)
    )
    ** 2
) ** 1.0 / (
    (bp1 * q1 + bp2 * q4 + bp3 * q7 + bp4 * q10) ** 2
    + (bp1 * q2 + bp2 * q5 + bp3 * q8 + bp4 * q11) ** 2
    + (bp1 * q3 + bp2 * q6 + bp3 * q9 + bp4 * q12) ** 2
) ** 3.0


In [2]:
import numpy as np

# create random array inputs
ng = np.random.default_rng()
# control points
q_np = ng.random(12).astype(float)
# bernstein coefficients
bp_np = ng.random(4).astype(float)
bpp_np = ng.random(4).astype(float)

In [3]:
def bending_function(Q, Bp, Bpp):
    xp = Q.__array_namespace__()
    QM = xp.reshape(Q, (4, 3)).T

    yip = xp.vecdot(QM, Bp)
    yipp = xp.vecdot(QM, Bpp)
    num = xp.linalg.vector_norm(xp.cross(yip, yipp))
    den = xp.linalg.vector_norm(yip) ** 3
    return (num / den) ** 2


bending_function(q_np, bp_np, bpp_np)

# Try sympy on random inputs to make sure they agree
(
    bending_function_sympy(
        Array(q_np.tolist()),
        Array(bp_np.tolist()),
        Array(bpp_np.tolist()),
    ),
    bending_function(q_np, bp_np, bpp_np),
)

(0.00149802436508140, np.float64(0.0014980243650814028))

In [4]:
import egglog
import egglog.exp.array_api as enp

Bp = enp.NDArray([egglog.constant(f"bp{i}", enp.Value) for i in range(1, 5)])
Bpp = enp.NDArray([egglog.constant(f"bpp{i}", enp.Value) for i in range(1, 5)])
Q = enp.NDArray([egglog.constant(f"q{i}", enp.Value) for i in range(1, 13)])
res = bending_function(Q, Bp, Bpp)
res

_NDArray_1 = reshape(
    NDArray(
        RecursiveValue.vec(
            Vec(
                RecursiveValue(q1),
                RecursiveValue(q2),
                RecursiveValue(q3),
                RecursiveValue(q4),
                RecursiveValue(q5),
                RecursiveValue(q6),
                RecursiveValue(q7),
                RecursiveValue(q8),
                RecursiveValue(q9),
                RecursiveValue(q10),
                RecursiveValue(q11),
                RecursiveValue(q12),
            )
        )
    ),
    TupleInt(Vec(Int(4), Int(3))),
).T
_NDArray_2 = vecdot(
    _NDArray_1,
    NDArray(
        RecursiveValue.vec(
            Vec(
                RecursiveValue(bp1),
                RecursiveValue(bp2),
                RecursiveValue(bp3),
                RecursiveValue(bp4),
            )
        )
    ),
)
(
    vector_norm(
        cross(
            _NDArray_2,
            vecdot(
                _NDArray_1,
                NDArray(
                    RecursiveValue.vec(
                        Vec(
                            RecursiveValue(bpp1),
                            RecursiveValue(bpp2),
                            RecursiveValue(bpp3),
                            RecursiveValue(bpp4),
                        )
                    )
                ),
            ),
        )
    )
    / vector_norm(_NDArray_2) ** NDArray(RecursiveValue(Value.from_int(Int(3))))
) ** NDArray(RecursiveValue(Value.from_int(Int(2))))

In [5]:
scalar_res = res.eval()

In [6]:
from __future__ import annotations
from dataclasses import dataclass
from functools import total_ordering


@total_ordering
@dataclass(frozen=True)
class ArithmeticCost:
    """
    Cost type where *, +, and ** are the same and then / is more than all of them.
    """

    muls: int = 0
    divs: int = 0
    adds: int = 0
    subs: int = 0
    exps: int = 0

    @property
    def not_div_ops(self) -> int:
        return self.muls + self.adds + self.exps + self.subs

    def __eq__(self, other: ArithmeticCost) -> bool:
        return self.not_div_ops == other.not_div_ops and self.divs == other.divs

    def __gt__(self, other: ArithmeticCost) -> bool:
        if self.divs != other.divs:
            return self.divs > other.divs
        return self.not_div_ops > other.not_div_ops

    def __add__(self, other: ArithmeticCost) -> ArithmeticCost:
        return ArithmeticCost(
            muls=self.muls + other.muls,
            divs=self.divs + other.divs,
            adds=self.adds + other.adds,
            exps=self.exps + other.exps,
            subs=self.subs + other.subs,
        )

    def __sub__(self, other: ArithmeticCost) -> ArithmeticCost:
        return ArithmeticCost(
            muls=self.muls - other.muls,
            divs=self.divs - other.divs,
            adds=self.adds - other.adds,
            exps=self.exps - other.exps,
            subs=self.subs - other.subs,
        )


def arith_cost_model(egraph: egglog.EGraph, expr: egglog.Expr, children_costs: list[ArithmeticCost]) -> ArithmeticCost:
    # named variables are free
    if egglog.get_constant_name(expr) is not None:
        return ArithmeticCost()
    # literals are free
    if egglog.get_literal_value(expr) is not None:
        return ArithmeticCost()
    match egglog.get_callable_fn(expr):
        case enp.Value.__add__:
            c = ArithmeticCost(adds=1)
        case enp.Value.__sub__:
            c = ArithmeticCost(subs=1)
        case enp.Value.__mul__:
            c = ArithmeticCost(muls=1)
        case enp.Value.__truediv__:
            c = ArithmeticCost(divs=1)
        case enp.Value.__pow__:
            c = ArithmeticCost(exps=1)
        case enp.Int | enp.Value.from_int:
            c = ArithmeticCost()
        case _ as fn:
            raise NotImplementedError(f"Unknown expr for arith cost model: {fn or expr}")
    return sum(children_costs, c)


egraph = egglog.EGraph()
egraph.register(scalar_res)
_, original_cost = egraph.extract(scalar_res, include_cost=True, cost_model=arith_cost_model)
print(original_cost)
scalar_res

ArithmeticCost(muls=66, divs=1, adds=49, subs=3, exps=7)


_Value_1 = q2 * bp1 + q5 * bp2 + q8 * bp3 + q11 * bp4
_Value_2 = q3 * bpp1 + q6 * bpp2 + q9 * bpp3 + q12 * bpp4
_Value_3 = q3 * bp1 + q6 * bp2 + q9 * bp3 + q12 * bp4
_Value_4 = q2 * bpp1 + q5 * bpp2 + q8 * bpp3 + q11 * bpp4
_Value_5 = q1 * bpp1 + q4 * bpp2 + q7 * bpp3 + q10 * bpp4
_Value_6 = q1 * bp1 + q4 * bp2 + q7 * bp3 + q10 * bp4
(
    (_Value_1 * _Value_2 - _Value_3 * _Value_4) ** Value.from_int(Int(2))
    + (_Value_3 * _Value_5 - _Value_6 * _Value_2) ** Value.from_int(Int(2))
    + (_Value_6 * _Value_4 - _Value_1 * _Value_5) ** Value.from_int(Int(2))
) / (
    _Value_6 ** Value.from_int(Int(2))
    + _Value_1 ** Value.from_int(Int(2))
    + _Value_3 ** Value.from_int(Int(2))
) ** Value.from_int(
    Int(3)
)

Now let's try distributing...

In [7]:
@egglog.ruleset
def distribute(a: enp.Value, b: enp.Value, c: enp.Value):
    yield egglog.rewrite((a + b) * c, subsume=True).to(a * c + b * c)
    yield egglog.rewrite(c * (a + b), subsume=True).to(c * a + c * b)
    yield egglog.rewrite((a - b) * c, subsume=True).to(a * c - b * c)
    yield egglog.rewrite(c * (a - b), subsume=True).to(c * a - c * b)
    # Push down subtraction
    yield egglog.rewrite(a - (b + c), subsume=True).to(a - b - c)
    yield egglog.rewrite(a - (b - c), subsume=True).to(a - b + c)


egraph.run(distribute.saturate())
distributed_res, distributed_cost = egraph.extract(scalar_res, cost_model=arith_cost_model, include_cost=True)
print(distributed_cost)
distributed_res

ArithmeticCost(muls=300, divs=1, adds=58, subs=48, exps=7)


(
    (
        q2 * bp1 * (q3 * bpp1)
        + q2 * bp1 * (q6 * bpp2)
        + q2 * bp1 * (q9 * bpp3)
        + q2 * bp1 * (q12 * bpp4)
        + (
            q5 * bp2 * (q3 * bpp1)
            + q5 * bp2 * (q6 * bpp2)
            + q5 * bp2 * (q9 * bpp3)
            + q5 * bp2 * (q12 * bpp4)
        )
        + (
            q8 * bp3 * (q3 * bpp1)
            + q8 * bp3 * (q6 * bpp2)
            + q8 * bp3 * (q9 * bpp3)
            + q8 * bp3 * (q12 * bpp4)
        )
        + (
            q11 * bp4 * (q3 * bpp1)
            + q11 * bp4 * (q6 * bpp2)
            + q11 * bp4 * (q9 * bpp3)
            + q11 * bp4 * (q12 * bpp4)
        )
        - q3 * bp1 * (q2 * bpp1)
        - q6 * bp2 * (q2 * bpp1)
        - q3 * bp1 * (q5 * bpp2)
        - q6 * bp2 * (q5 * bpp2)
        - q3 * bp1 * (q8 * bpp3)
        - q6 * bp2 * (q8 * bpp3)
        - q3 * bp1 * (q11 * bpp4)
        - q6 * bp2 * (q11 * bpp4)
        - q9 * bp3 * (q2 * bpp1)
        - q9 * bp3 * (q5 * bpp2)
        - q9 * bp3 * (q8 * bpp3)
        - q9 * bp3 * (q11 * bpp4)
        - q12 * bp4 * (q2 * bpp1)
        - q12 * bp4 * (q5 * bpp2)
        - q12 * bp4 * (q8 * bpp3)
        - q12 * bp4 * (q11 * bpp4)
    )
    ** Value.from_int(Int(2))
    + (
        q3 * bp1 * (q1 * bpp1)
        + q3 * bp1 * (q4 * bpp2)
        + q3 * bp1 * (q7 * bpp3)
        + q3 * bp1 * (q10 * bpp4)
        + (
            q6 * bp2 * (q1 * bpp1)
            + q6 * bp2 * (q4 * bpp2)
            + q6 * bp2 * (q7 * bpp3)
            + q6 * bp2 * (q10 * bpp4)
        )
        + (
            q9 * bp3 * (q1 * bpp1)
            + q9 * bp3 * (q4 * bpp2)
            + q9 * bp3 * (q7 * bpp3)
            + q9 * bp3 * (q10 * bpp4)
        )
        + (
            q12 * bp4 * (q1 * bpp1)
            + q12 * bp4 * (q4 * bpp2)
            + q12 * bp4 * (q7 * bpp3)
            + q12 * bp4 * (q10 * bpp4)
        )
        - q1 * bp1 * (q3 * bpp1)
        - q4 * bp2 * (q3 * bpp1)
        - q1 * bp1 * (q6 * bpp2)
        - q4 * bp2 * (q6 * bpp2)
        - q1 * bp1 * (q9 * bpp3)
        - q4 * bp2 * (q9 * bpp3)
        - q1 * bp1 * (q12 * bpp4)
        - q4 * bp2 * (q12 * bpp4)
        - q7 * bp3 * (q3 * bpp1)
        - q7 * bp3 * (q6 * bpp2)
        - q7 * bp3 * (q9 * bpp3)
        - q7 * bp3 * (q12 * bpp4)
        - q10 * bp4 * (q3 * bpp1)
        - q10 * bp4 * (q6 * bpp2)
        - q10 * bp4 * (q9 * bpp3)
        - q10 * bp4 * (q12 * bpp4)
    )
    ** Value.from_int(Int(2))
    + (
        q1 * bp1 * (q2 * bpp1)
        + q1 * bp1 * (q5 * bpp2)
        + q1 * bp1 * (q8 * bpp3)
        + q1 * bp1 * (q11 * bpp4)
        + (
            q4 * bp2 * (q2 * bpp1)
            + q4 * bp2 * (q5 * bpp2)
            + q4 * bp2 * (q8 * bpp3)
            + q4 * bp2 * (q11 * bpp4)
        )
        + (
            q7 * bp3 * (q2 * bpp1)
            + q7 * bp3 * (q5 * bpp2)
            + q7 * bp3 * (q8 * bpp3)
            + q7 * bp3 * (q11 * bpp4)
        )
        + (
            q10 * bp4 * (q2 * bpp1)
            + q10 * bp4 * (q5 * bpp2)
            + q10 * bp4 * (q8 * bpp3)
            + q10 * bp4 * (q11 * bpp4)
        )
        - q2 * bp1 * (q1 * bpp1)
        - q5 * bp2 * (q1 * bpp1)
        - q2 * bp1 * (q4 * bpp2)
        - q5 * bp2 * (q4 * bpp2)
        - q2 * bp1 * (q7 * bpp3)
        - q5 * bp2 * (q7 * bpp3)
        - q2 * bp1 * (q10 * bpp4)
        - q5 * bp2 * (q10 * bpp4)
        - q8 * bp3 * (q1 * bpp1)
        - q8 * bp3 * (q4 * bpp2)
        - q8 * bp3 * (q7 * bpp3)
        - q8 * bp3 * (q10 * bpp4)
        - q11 * bp4 * (q1 * bpp1)
        - q11 * bp4 * (q4 * bpp2)
        - q11 * bp4 * (q7 * bpp3)
        - q11 * bp4 * (q10 * bpp4)
    )
    ** Value.from_int(Int(2))
) / (
    (q1 * bp1 + q4 * bp2 + q7 * bp3 + q10 * bp4) ** Value.from_int(Int(2))
    + (q2 * bp1 + q5 * bp2 + q8 * bp3 + q11 * bp4) ** Value.from_int(Int(2))
    + (q3 * bp1 + q6 * bp2 + q9 * bp3 + q12 * bp4) ** Value.from_int(Int(2))
) ** Value.from_int(
    Int(3)
)

In [8]:
egraph = egglog.EGraph()
egraph.register(distributed_res)
egraph.run(enp.polynomial_schedule)
factored_res, factored_cost = egraph.extract(distributed_res, include_cost=True, cost_model=arith_cost_model)
print(factored_cost)
factored_res

ArithmeticCost(muls=159, divs=1, adds=106, subs=0, exps=7)


_Value_1 = bp2 * q5 + q2 * bp1 + bp3 * q8 + q11 * bp4
_Value_2 = q2 * bpp1 + q11 * bpp4 + q5 * bpp2 + q8 * bpp3
_Value_3 = bpp1 * q1 + q4 * bpp2 + q7 * bpp3 + bpp4 * q10
_Value_4 = q4 * bp2 + q1 * bp1 + q7 * bp3 + q10 * bp4
(
    (
        q12 * (bpp4 * _Value_1)
        + q9 * (bpp3 * _Value_1)
        + q6 * (bpp2 * _Value_1)
        + Value.from_int(Int(-1))
        * (
            q6 * (bp2 * _Value_2)
            + q9 * (bp3 * _Value_2)
            + q3 * (bp1 * _Value_2)
            + q12 * (bp4 * _Value_2)
        )
        + q3 * (bpp1 * _Value_1)
    )
    ** Value.from_int(Int(2))
    + (
        q6 * (bp2 * _Value_3)
        + q9 * (bp3 * _Value_3)
        + q12 * (bp4 * _Value_3)
        + Value.from_int(Int(-1))
        * (
            q3 * (bpp1 * _Value_4)
            + q6 * (bpp2 * _Value_4)
            + q9 * (bpp3 * _Value_4)
            + q12 * (bpp4 * _Value_4)
        )
        + q3 * (bp1 * _Value_3)
    )
    ** Value.from_int(Int(2))
    + (
        bp2 * (q4 * _Value_2)
        + bp3 * (q7 * _Value_2)
        + bp4 * (q10 * _Value_2)
        + bp1 * (q1 * _Value_2)
        + Value.from_int(Int(-1))
        * (
            bp1 * (q2 * _Value_3)
            + q8 * (bp3 * _Value_3)
            + q5 * (bp2 * _Value_3)
            + bp4 * (q11 * _Value_3)
        )
    )
    ** Value.from_int(Int(2))
) / (
    _Value_4 ** Value.from_int(Int(2))
    + _Value_1 ** Value.from_int(Int(2))
    + (bp1 * q3 + bp2 * q6 + bp3 * q9 + bp4 * q12) ** Value.from_int(Int(2))
) ** Value.from_int(
    Int(3)
)

In [9]:
original_cost.not_div_ops

125

In [10]:
factored_cost.not_div_ops

272

In [11]:
distributed_cost.not_div_ops

413